# LLM을 사용하며 알아야 할 것들

이 노트북은 슬라이드와 설명, 퀴즈로 개념을 익히는 강의 노트북입니다. 코드 실습은 `1-2_llm_api_lab.ipynb`에서 합니다.

**목차**

- 1. LLM의 학습과 추론 (슬라이드) (퀴즈)
  - 1.1 사전 학습과 Instruction Tuning (슬라이드) (퀴즈)
  - 1.2 다음 토큰 예측과 생성 과정 (슬라이드) (퀴즈)
- 2. Tokenization (슬라이드) (퀴즈)
  - 2.1 토큰의 의미 (슬라이드) (퀴즈)
  - 2.2 토큰 수와 비용 (슬라이드) (퀴즈)
- 3. Context Window (슬라이드)
  - 3.1 Context Window (슬라이드)
  - 3.2 입력 품질과 출력 품질 (슬라이드)
- 4. LLM API의 대화 구조 (슬라이드) (퀴즈)
  - 4.1 System, User, Assistant 역할 (슬라이드) (퀴즈)
  - 4.2 모델 파라미터의 영향 (슬라이드) (퀴즈)
- 5. 정리 퀴즈 (퀴즈)

In [ ]:
import json
import sys

sys.path.insert(0, '..')  # lessons/assets 도구를 불러오기 위한 경로

from assets.widgets import show_order_quiz, show_quiz, show_slides, slide_setup

slide_setup()  # 슬라이드 뷰어(PDF.js)를 불러옵니다. 슬라이드가 비어 보이면 이 셀을 다시 실행합니다.

In [ ]:
show_slides("1-2", [1, 2])

이번 시간은 LLM을 쓸 때 알아야 할 네 가지를 차례로 봅니다.

1. LLM의 학습과 추론: 모델이 어떻게 배우고, 어떻게 답을 만드는가
2. Tokenization(토큰화: 글을 모델이 다루는 작은 조각으로 나누는 일): 토큰의 뜻과 비용
3. Context Window(컨텍스트 윈도우: 모델이 한 번에 읽는 범위): 읽는 범위와 입력 품질
4. LLM API의 대화 구조: 역할(System·User·Assistant)과 모델 파라미터

이 노트북은 개념만 다룹니다. 여기서 본 내용을 코드로 호출해 보는 일은 실습 노트북에서 합니다.

**왜 배우나:** 뒤에 나오는 LangChain·RAG·LangGraph·캡스톤은 모두 LLM API 호출 위에 쌓입니다. 토큰, 컨텍스트 윈도우, 메시지 역할을 알면 그 코드가 왜 그렇게 짜였는지 읽을 수 있습니다.

<details><summary>강사용 메모</summary>

| 항목 | 내용 |
|---|---|
| 버전 | C |
| 원본 PDF | `docs/raw/Chapter01/day01/1-2-development-environment-and-llm-api-mechanism.pdf` (19쪽) |
| 원본 코드 | 이 노트북에는 없습니다. `src/raw/Chapter01/01-llm-api/`의 세 파일은 실습 노트북 `1-2_llm_api_lab.ipynb`에서 다룹니다. |
| API 호출 | 없습니다. 슬라이드 준비 셀만 실행하면 됩니다. |

---

실습 코드(`01_requests.py`, `02_openai.py`, `03_langchain.py` 실행, CodeMate 실습)는 `1-2_llm_api_lab.ipynb`에 있습니다. 이 노트북은 코드 셀 없이 슬라이드·설명·퀴즈만 담습니다.

</details>

## 1. LLM의 학습과 추론 <small>(슬라이드) (퀴즈)</small>

### 1.1 사전 학습과 Instruction Tuning <small>(슬라이드) (퀴즈)</small>

슬라이드를 보기 전에 먼저 예측해 봅니다. 답은 바로 다음 슬라이드에 있습니다.

In [ ]:
show_quiz("../assets/quizzes/1-2/q11.json")

In [ ]:
show_slides("1-2", [3, 4])

LLM은 먼저 아주 많은 글(웹 문서, 책, 코드, 대화)에서 문장이 이어지는 패턴을 배웁니다.
이것을 사전 학습(Pre-training)이라고 합니다.
이 단계의 목표는 "지금까지 나온 글 다음에 올 가능성이 높은 토큰(글 조각)"을 맞히는 것입니다.
그래서 사전 학습만 한 모델은 "To be or not to be"를 질문으로 보지 않고, "that is the question"을 이어 씁니다.
이 "다음 토큰 맞히기"는 1.2에서 답이 토큰 하나씩 만들어지는 과정으로 이어집니다.

In [ ]:
show_slides("1-2", [5])

Instruction Tuning(지시 학습)은 모델이 사용자의 입력을 "해야 할 요청"으로 이해하도록 더 가르치는 과정입니다.
질문과 답, 명령과 결과, 대화 예시 같은 데이터로 추가 학습합니다.
슬라이드 예시처럼 `instruction`(지시), `input`(입력), `output`(기대하는 답)을 한 묶음으로 보여 줍니다.
API나 챗봇에서 쓰는 모델은 대체로 이 과정을 거친 모델입니다.

**왜 배우나:** API 모델은 지시를 따르도록 조정된 모델이라서, 지시를 어떻게 쓰느냐에 따라 결과가 달라집니다.
CodeMate 같은 Coding AI에게 일을 시킬 때도 같습니다. 무엇을 원하는지 분명하게 지시하면 뭉뚱그린 질문보다 결과가 좋습니다.

**정리:** 사전 학습은 이어 쓰기를 배우고, Instruction Tuning은 지시 따르기를 배웁니다.

### 1.2 다음 토큰 예측과 생성 과정 <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("1-2", [6])

LLM은 완성된 답을 한 번에 꺼내지 않습니다.
지금까지의 글을 보고 다음에 올 토큰들의 확률(가능성)을 계산하고, 그중 하나를 골라 뒤에 붙입니다.
이것을 반복하면 단어, 문장, 문단, 최종 답이 차례로 만들어집니다.

**왜 배우나:** 답이 조각으로 만들어진다는 것을 알아야 Streaming(스트리밍: 답이 다 만들어지기 전에 조각이 생기는 대로 받는 방식)을 이해할 수 있습니다.
스트리밍은 2일차에 RAG 체인의 응답과 Streamlit 챗봇 화면을 만들 때 다시 씁니다. 답이 하나씩 붙어 출력되는 모습은 실습 노트북에서 직접 확인합니다.

**먼저 예측해 봅니다.** 다음 쪽을 보기 전에, 요청을 보내고 응답을 받기까지의 순서를 맞혀 보세요.

In [ ]:
show_order_quiz(**json.load(open("../assets/quizzes/1-2/order12.json", encoding="utf-8")))

In [ ]:
show_slides("1-2", [7])

LLM API(프로그램끼리 요청과 답을 주고받는 창구)는 앞에서 본 반복 생성을 네 단계로 감쌉니다.

1. 사용자가 메시지와 설정값을 요청으로 보냅니다.
2. 서버의 모델이 입력을 읽고 다음 토큰 후보를 계산합니다.
3. 고른 토큰이 쌓여 답 메시지가 됩니다.
4. 끝낼 조건에 닿으면 최종 답을 응답으로 돌려줍니다.

호출하는 쪽에서 보이는 것은 1단계(요청)와 4단계(응답)뿐입니다. 2·3단계는 서버 안에서 일어납니다.

**왜 배우나:** 뒤에서 LangChain으로 하는 호출도 속에서는 이 요청과 응답을 주고받습니다. 이 네 단계를 알면 호출이 실패했을 때 어느 단계에서 막혔는지 나눠 볼 수 있습니다.

**정리:** 모델은 토큰을 하나씩 골라 붙이고, API는 이 과정을 "요청 → 계산 → 누적 → 반환"으로 감쌉니다.

## 2. Tokenization <small>(슬라이드) (퀴즈)</small>

### 2.1 토큰의 의미 <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("1-2", [8, 9])

Tokenization은 사람의 말을 모델이 다룰 수 있는 작은 단위로 나누는 과정이고, 나뉜 단위를 토큰이라고 부릅니다.
토큰은 단어와 항상 같지 않습니다. 단어의 일부나 공백·기호·숫자 조각이 될 수 있습니다.
한국어는 조사와 어미, 띄어쓰기 때문에 더 잘게 나뉠 수 있습니다. 예시에서 "를", "다", "." 같은 조각이 따로 떨어진 것을 보세요.
모델은 글자를 그대로 받지 않고 토큰마다 숫자 묶음(벡터)으로 바꾼 것을 입력받습니다.

**왜 배우나:** 입력 프롬프트와 모델의 답은 모두 토큰 단위로 셉니다.
그래서 토큰을 알아야 바로 뒤의 비용(2.2)과 컨텍스트 윈도우 크기(3.1)를 이해할 수 있습니다.

In [ ]:
show_quiz("../assets/quizzes/1-2/q21.json")

**정리:** 토큰은 모델이 읽고 쓰는 기본 단위이고, 단어와 같지 않습니다.

### 2.2 토큰 수와 비용 <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("1-2", [10])

LLM API의 비용과 처리량은 보통 입력 토큰 수와 출력 토큰 수로 정해집니다.
입력 토큰에는 입력 프롬프트, 시스템 프롬프트, 이전 대화 내용처럼 모델이 답하는 데 쓰는 모든 것이 들어갑니다.
출력 토큰은 모델이 새로 만든 답입니다. 추론 기능이 있는 모델이라면 추론하며 만든 토큰도 출력에 들어갑니다.
프롬프트와 답이 길수록 비용이 늘고 응답 시간도 길어집니다.

**왜 배우나:** 대화가 길어지면 이전 대화가 매번 입력으로 다시 들어가서 한 턴마다 비용이 커집니다.
Coding AI를 쓸 때도 같습니다. 분명하게 지시하면 엉뚱한 재시도로 버리는 토큰이 줄어듭니다.

In [ ]:
show_slides("1-2", [11])

토큰 수를 줄여 비용을 아끼는 방법은 네 가지입니다.

1. 필요한 정보만 골라서 넣습니다.
2. 반복되는 지시문과 대화 기록을 줄여 씁니다.
3. 검색한 문서를 다 넣지 않고 관련 높은 일부만 넣습니다.
4. 답의 형식과 길이를 분명히 정해 쓸데없는 출력을 줄입니다. 예를 들어 "3문장으로 설명해줘"처럼 길이를 정합니다.

**왜 배우나:** 3번 "관련 높은 일부만 넣기"는 2일차 RAG에서 검색(Retrieval) 단계가 하는 일입니다. 2번 "대화 기록 줄이기"는 대화 요약과 메모리 설계로 이어집니다.

In [ ]:
show_quiz("../assets/quizzes/1-2/q22.json")

**정리:** 비용은 입력 토큰(프롬프트·시스템 지시·이전 대화)과 출력 토큰(새 답, 추론 토큰)으로 정해집니다.

## 3. Context Window <small>(슬라이드)</small>

### 3.1 Context Window <small>(슬라이드)</small>

In [ ]:
show_slides("1-2", [12, 13, 14])

Context Window는 모델이 한 번의 요청에서 읽고 고려할 수 있는 토큰 범위입니다.
이 범위에는 시스템 메시지, 사용자 메시지, 이전 대화, 도구 실행 결과가 모두 들어갑니다.
모델은 이번 요청에 들어 있지 않은 정보를 직접 볼 수 없습니다.
그래서 컨텍스트 윈도우가 LLM 애플리케이션 설계의 한계를 정합니다.
긴 문서는 나눠서 필요한 부분만 검색해 넣고, 긴 대화는 요약하거나 중요한 상태만 따로 저장합니다.
Agent가 여러 도구를 쓸 때는 다음 판단에 필요한 정보만 남깁니다.

**왜 배우나:** 모델은 기억하지 않으므로 대화를 이어 가려면 매번 이전 대화를 요청에 다시 넣어야 하고, 대화가 길수록 매 턴 입력 토큰과 비용이 커집니다.
그래서 대화 요약, 2일차의 RAG, LangGraph의 상태(state: 그래프가 들고 다니는 정보 묶음) 설계가 필요합니다. 컨텍스트 윈도우를 알아야 Agent에 무엇을 넣고 뺄지 설계할 수 있습니다.

**정리:** 모델은 이번 요청에 담긴 것만 봅니다. 기억시키려면 다시 넣어야 합니다.

### 3.2 입력 품질과 출력 품질 <small>(슬라이드)</small>

In [ ]:
show_slides("1-2", [15, 16])

LLM의 답 품질은 모델 성능만이 아니라 컨텍스트에 어떤 정보를 넣었는지에 크게 좌우됩니다.
목표가 분명하면 답의 방향이 안정되고, 배경 정보가 충분하면 추측과 할루시네이션이 줄어듭니다.
쓸데없거나 서로 부딪히는 정보가 많으면 모델의 판단이 흐려질 수 있습니다.
좋은 프롬프트(모델에게 주는 지시문)는 답을 맞히라고 요구하지 않고, 답을 만들 수 있는 조건을 줍니다.
슬라이드 아래쪽의 네 항목(역할·목표, 배경·제약, 출력 형식·평가 기준, 모호함 줄이기)이 그 조건입니다.

**왜 배우나:** 이 네 항목은 CodeMate 같은 Coding AI에게 일을 시킬 때도 지시문의 뼈대가 됩니다.
Coding AI가 코드를 써 주더라도, 무엇을 어떤 조건으로 만들지 정하는 지시는 사람이 씁니다.

**정리:** 좋은 입력은 역할·목표, 배경·제약, 출력 형식, 모호함 줄이기의 네 조건을 줍니다.

## 4. LLM API의 대화 구조 <small>(슬라이드) (퀴즈)</small>

### 4.1 System, User, Assistant 역할 <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("1-2", [17, 18])

Chat 기반 LLM API는 대화를 역할이 붙은 메시지 목록으로 나타냅니다.
System 메시지는 모델의 전체 역할, 규칙, 제약 조건을 정합니다.
User 메시지는 사용자의 질문, 요청, 입력 데이터를 전합니다.
Assistant 메시지는 LLM이 만든 답이고, 다음 대화에 추가되어 맥락을 전합니다.
역할 구분은 모양만이 아니라 모델이 지시의 우선순위와 대화 흐름을 읽는 기준입니다.

**왜 배우나:** 4일차 Harness 실습에서는 "범위를 벗어난 요청 거절"을 System 프롬프트로 만듭니다. 이때 System 메시지도 매번 입력 토큰이 된다는 점을 함께 기억합니다.

In [ ]:
show_quiz("../assets/quizzes/1-2/q41.json")

**정리:** System은 규칙, User는 요청, Assistant는 모델의 답이며, 이 구분이 지시의 우선순위를 정합니다.

### 4.2 모델 파라미터의 영향 <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("1-2", [19])

모델 파라미터(모델에 함께 넘기는 설정값)는 같은 입력에서도 답이 얼마나 다양하고 길고 예측 가능할지를 조정합니다.
Temperature는 다음에 올 수 있는 토큰들의 확률을 조정하고, Top-p는 후보 토큰들의 누적 확률(확률이 높은 후보부터 더해 간 값) 한도를 조정합니다.
낮은 값은 일관되고 보수적인 답에, 높은 값은 다양한 표현과 아이디어에 유리합니다.
너무 높으면 전혀 가능성 없는 토큰이 나올 수 있어 조심합니다.

**왜 배우나:** 1.2에서 모델이 "후보 중 하나를 고른다"고 했는데, 그 고르는 방식을 사람이 조정하는 손잡이가 이 두 값입니다.
같은 질문을 다시 보내도 답이 조금씩 다를 수 있는 이유이기도 합니다.

In [ ]:
show_quiz("../assets/quizzes/1-2/q42.json")

**정리:** Temperature와 Top-p를 낮추면 일관된 답, 높이면 다양한 답이 나옵니다.

## 5. 정리 퀴즈 <small>(퀴즈)</small>

In [ ]:
show_quiz("../assets/quizzes/1-2/final.json")

**이 노트북에서 본 것**
- 사전 학습만 한 모델은 그럴듯하게 이어 쓰고 API로 쓰는 모델은 대체로 지시 수행을 강화한 모델입니다.
- 모델은 토큰을 하나씩 예측해 붙이고, API는 요청 → 후보 계산 → 누적 → 반환 순서로 답을 돌려줍니다.
- 비용은 입력 토큰과 출력 토큰으로 정해지고, 입력에는 시스템 프롬프트와 이전 대화까지 들어갑니다.
- 모델은 요청에 담긴 컨텍스트만 읽습니다. 이전 대화도 요청에 다시 넣어야 합니다.
- 좋은 입력은 역할·목표, 배경·제약, 출력 형식을 주고 모호함을 줄입니다.
- 역할 구분(System, User, Assistant)은 모델이 지시의 우선순위와 대화 흐름을 해석하는 기준입니다.
- Temperature와 Top-p가 낮으면 일관된 답, 높으면 다양한 답에 유리합니다.